# Reluplex and MILP on an ACAS Xu–style property — in-class demo

**998MG Safe and Verified AI — 2026/27 · Lecture 8 · Exercise 2 (slide 59)**

Two inputs $\rho \in [50,100]$, $v \in [0,20]$, one output $y$:

```
z1 = ρ + v − 10          a1 = relu(z1)
z2 = 2ρ − 30v − 80       a2 = relu(z2)
y  = 4·a1 + 5·a2 + 1000
```

**Property:** if $\rho \ge 50$ and $v \le 20$, the clear-of-conflict score is $y \le 1500$.

We answer the four questions of the slide with two tools:

| tool | what it answers | lecture |
|---|---|---|
| **Z3** (SMT) | *is there* an input with $y > 1500$? — the satisfiability query | L07 |
| **SciPy** `milp` (HiGHS inside) | *what is the largest* $y$ on the box? — and its LP relaxation | L08 |

Everything runs in a few seconds.

In [ ]:
# Colab only (locally, requirements.txt has already installed it): uncomment the next line
# %pip install -q z3-solver

In [ ]:
# numpy and matplotlib for the picture, SciPy for MILP and LP, Z3 for SMT
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import milp, LinearConstraint, Bounds
import z3

## The network and the property

We write the network once, as weights, so that every method below reads the same numbers.
`net` evaluates it exactly. We use it to **check** every counterexample a solver hands us.

In [ ]:
# hidden layer: z = W·(rho, v) + B
W  = [[1,   1],          # z1 = 1·ρ +  1·v − 10
      [2, -30]]          # z2 = 2·ρ − 30·v − 80
B  = [-10, -80]
# output layer: y = 4·a1 + 5·a2 + 1000
C  = [4, 5]
C0 = 1000

RHO, V    = (50, 100), (0, 20)   # the input box
THRESHOLD = 1500                 # the property: y <= 1500

# run the network on one input; returns the pre-activations (z1, z2) and the output y
def net(rho, v):
    z = [W[i][0]*rho + W[i][1]*v + B[i] for i in range(2)]
    a = [max(zi, 0) for zi in z]
    return z, C[0]*a[0] + C[1]*a[1] + C0

net(100, 0)

## Step 1 — bounds of $z_1$ and $z_2$ on the input box

Interval arithmetic: for $z = \sum_j w_j x_j + b$, the lower bound takes each $x_j$ at its
lower end when $w_j \ge 0$ and at its upper end when $w_j < 0$; the upper bound does the opposite.

In [ ]:
# bounds of one affine function w·x + b when each x_j ranges over the interval box[j]
def affine_bounds(w, b, box):
    lo = b + sum(wj * (l if wj >= 0 else u) for wj, (l, u) in zip(w, box))
    hi = b + sum(wj * (u if wj >= 0 else l) for wj, (l, u) in zip(w, box))
    return lo, hi

# [l1, u1] and [l2, u2] for the two hidden neurons
def preactivation_bounds(rho_box, v_box):
    return [affine_bounds(W[i], B[i], [rho_box, v_box]) for i in range(2)]

bounds = preactivation_bounds(RHO, V)
for i, (l, u) in enumerate(bounds, start=1):
    print(f"z{i} in [{l}, {u}]")

## Step 2 — which ReLU is stable?

A ReLU is **stable** when its input never changes sign on the box: $l \ge 0$ (always active,
$a = z$) or $u \le 0$ (always inactive, $a = 0$). Otherwise it is **unstable**.

In [ ]:
def phase(l, u):
    if l >= 0: return "stable, always active  (a = z)"
    if u <= 0: return "stable, always inactive (a = 0)"
    return "UNSTABLE"

for i, (l, u) in enumerate(bounds, start=1):
    print(f"relu(z{i}): {phase(l, u)}")

# the same intervals give a first, cheap bound on y: the output weights c_i are > 0,
# so y is largest when every a_i = relu(z_i) is at its largest value, max(u_i, 0)
y_interval = C0 + sum(c * max(u, 0) for c, (l, u) in zip(C, bounds))
print(f"\ninterval bound: y <= {y_interval}  ->  > {THRESHOLD}, so intervals alone cannot decide")

### What the input box looks like

Colour is $y$. The red line is $y = 1500$: everything below it and to its right violates the
property. The dashed line is $z_2 = 0$, where the unstable ReLU switches. Every violation lies on
the side where it is active.

In [ ]:
# y on a 401 x 401 grid of the box
r = np.linspace(*RHO, 401)
s = np.linspace(*V, 401)
R, S = np.meshgrid(r, s)
Z2 = 2*R - 30*S - 80
Y = 4*np.maximum(R + S - 10, 0) + 5*np.maximum(Z2, 0) + 1000

# one hue, light -> dark, for the magnitude of y
teal = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "teal", ["#EDF2F3", "#C9DEE5", "#4FA3B4", "#10697C", "#0B3F4A"])

fig, ax = plt.subplots(figsize=(7.5, 4))
im = ax.pcolormesh(R, S, Y, cmap=teal, shading="auto")
ax.contour(R, S, Y, levels=[THRESHOLD], colors="#C2402A", linewidths=2)
ax.contour(R, S, Z2, levels=[0], colors="#4A5A61", linewidths=1.5, linestyles="--")
ax.plot(100, 0, "o", ms=9, color="#C2402A", mec="white", mew=2, clip_on=False, zorder=5)
ax.text(66, 8.5, "relu(z2) inactive here:  y ≤ 1440", color="#1B2B33")

# contour lines carry no label of their own, so we build the legend by hand
ax.legend(handles=[
    Line2D([], [], color="#C2402A", lw=2, label="y = 1500  (violations below / right)"),
    Line2D([], [], color="#4A5A61", lw=1.5, ls="--", label="z2 = 0  (relu(z2) active below)"),
    Line2D([], [], color="#C2402A", marker="o", ls="", ms=8, mec="white", mew=2,
           label="max y = 1960  at ρ = 100, v = 0")],
    loc="upper left", frameon=False)

ax.set_xlabel("ρ"); ax.set_ylabel("v")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.colorbar(im, ax=ax, label="y", fraction=0.04, pad=0.02)
plt.tight_layout(); plt.show()

## Step 3a — the satisfiability query (Z3)

To prove $P$ we ask for a model of $\neg P$: an input **in the box** whose output satisfies
**$y > 1500$**.

- `unsat` → no such input: the property **holds**;
- `sat` → the model **is** a counterexample.

`If(z >= 0, z, 0)` is the ReLU. Z3 is a general SMT solver, **not Reluplex**: its SAT engine
splits on the condition `z >= 0` of every `If`, and a simplex checks each branch (DPLL(T), L07).

In [ ]:
# real-valued unknowns: the two inputs and every neuron
rho, v = z3.Reals("rho v")
z1, z2, a1, a2, y = z3.Reals("z1 z2 a1 a2 y")
relu = lambda z: z3.If(z >= 0, z, 0)

box     = [rho >= RHO[0], rho <= RHO[1], v >= V[0], v <= V[1]]
network = [z1 == rho + v - 10,
           z2 == 2*rho - 30*v - 80,
           a1 == relu(z1),
           a2 == relu(z2),
           y  == 4*a1 + 5*a2 + 1000]

solver = z3.Solver()
solver.add(box + network)
solver.add(y > THRESHOLD)          # the NEGATED property
print(solver.check())

In [ ]:
# Z3 answers in exact rationals: read them as Python fractions
m = solver.model()
cex = {name: m[var].as_fraction() for name, var in [("rho", rho), ("v", v)]}
print("counterexample:", {k: f"{x} ≈ {float(x):.2f}" for k, x in cex.items()})

# never trust a counterexample you have not re-run: evaluate it exactly
(z1_, z2_), y_ = net(cex["rho"], cex["v"])
print(f"z1 = {float(z1_):.2f},  z2 = {float(z2_):.2f},  y = {y_}  > {THRESHOLD}: {y_ > THRESHOLD}")

Z3 returns **a** counterexample, the first one it finds, just above the threshold. It is a
witness, not the worst case. We come back to the worst case with optimisation.

### What Reluplex does instead: derive bounds, then split

Reluplex (Katz et al., CAV 2017) is not Z3. The basic algorithm of the slides (simplex, repairs,
splits) only uses the bounds of the query and of the splits. The implementation adds one more
ingredient (Section 4 of the paper, the rules *deriveLowerBound* and *deriveUpperBound*): as the
search goes, it **derives tighter bounds** from the equations of the tableau. For a row
$x_i = \sum_j T_{ij}\, x_j$, the lower bound of $x_i$ takes $l(x_j)$ where $T_{ij} > 0$ and $u(x_j)$ where
$T_{ij} < 0$: interval arithmetic, row by row. The paper applies the rule to the **basic** variable of each row,
so a bound reaches a variable only after a pivot has made it basic; the code below solves every row
for every variable at once. It is a faster version of the same rule, and every bound it finds is
still valid for the branch. When the bounds of a pair $x_f = \mathrm{relu}(x_b)$ show
$l(x_b) > 0$ or $l(x_f) > 0$, the ReLU is **fixed active**; when $u(x_b) < 0$, **fixed inactive**. A fixed
ReLU is just a linear equation and is never split. If some $l(x) > u(x)$, the branch is UNSAT.

The bounds include the **negated property**: $y > 1500$ is the bound $l(y) = 1500$, and it
propagates backwards through the rows like any other. We run the rules by hand, a few rounds.

In [ ]:
# the equations of the query, each written as  sum of coef·var + const = 0
rows = {"z1 row": ({"z1": 1, "rho": -1, "v": -1}, 10),           # z1 = rho + v − 10
        "z2 row": ({"z2": 1, "rho": -2, "v": 30}, 80),           # z2 = 2·rho − 30·v − 80
        "y row":  ({"y": 1, "a1": -4, "a2": -5}, -1000)}         # y  = 4·a1 + 5·a2 + 1000
pairs = [("z1", "a1"), ("z2", "a2")]                             # a = relu(z), not fixed yet
inf = np.inf
lb = {"rho": RHO[0], "v": V[0], "z1": -inf, "z2": -inf, "a1": 0,   "a2": 0,   "y": THRESHOLD}  # a >= 0; y > 1500
ub = {"rho": RHO[1], "v": V[1], "z1":  inf, "z2":  inf, "a1": inf, "a2": inf, "y": inf}

# deriveLowerBound / deriveUpperBound: solve a row for each of its variables and bound the other side
def derive_bounds():
    for name, (coef, const) in rows.items():
        for x, cx in coef.items():
            lo = hi = -const / cx
            for xj, cj in coef.items():
                if xj != x:
                    k = -cj / cx                                 # x = ... + k·xj + ...
                    lo += k * (lb[xj] if k > 0 else ub[xj])
                    hi += k * (ub[xj] if k > 0 else lb[xj])
            if lo > lb[x] + 1e-9: lb[x] = lo; print(f"  {name}:  {x} >= {lo:g}")
            if hi < ub[x] - 1e-9: ub[x] = hi; print(f"  {name}:  {x} <= {hi:g}")
            if lb[x] > ub[x]:     print(f"  {x} has l > u: UNSAT"); return

# a pair whose phase the bounds decide becomes a linear equation (or a = 0)
def fix_pairs():
    for z, a in list(pairs):
        if lb[z] > 0 or lb[a] > 0:
            print(f"  relu({z}) fixed ACTIVE:  l({z}) = {lb[z]:g}, l({a}) = {lb[a]:g}  ->  {a} = {z}")
            rows[f"{a} = {z}"] = ({a: 1, z: -1}, 0); pairs.remove((z, a))
        elif ub[z] < 0:
            print(f"  relu({z}) fixed INACTIVE:  u({z}) = {ub[z]:g}  ->  {a} = 0")
            ub[a] = 0; pairs.remove((z, a))

for r in (1, 2, 3):
    print(f"round {r}")
    derive_bounds()
    fix_pairs()
print("\nReLUs left to split:", pairs or "none")

Three rounds, **no split**:

1. the rows of $z_1$ and $z_2$ give the interval bounds of Step 1. $z_1 \ge 40 > 0$ fixes `relu(z1)`
   **active**: $a_1 = z_1$ becomes one more linear equation;
2. the new equation passes the bounds of $z_1$ to $a_1$: $a_1 \le 110$;
3. the row of $y$, with $y \ge 1500$, gives $5a_2 \ge 1500 - 1000 - 4\cdot 110 = 60$, so $a_2 \ge 12 > 0$:
   `relu(z2)` is fixed **active** too ($l(x_f) > 0$). Two more rounds give $z_2 \ge 12$, then $v \le 3.6$.

Every counterexample has both ReLUs active. The information came **backwards, from the property**:
the forward intervals of Step 1 leave $z_2$ unstable. More rounds keep shrinking the bounds, slowly
($\rho \ge 52.74$, $v \le 3.15$, $y \le 1972.6$ in the limit), so Reluplex does not run them to the end: it
tightens the entering variable after every pivot, and all the rows once every few thousand pivots.

Without bound derivation Reluplex would repair `relu(z2)` and, after $\tau$ failed repairs, split it.
We do that split by hand: one plain linear problem per phase. The inactive branch must be
UNSAT, as $a_2 \ge 12$ predicted.

In [ ]:
reluplex = z3.Solver()
reluplex.add(box)
reluplex.add(z1 == rho + v - 10, z2 == 2*rho - 30*v - 80,
             y == 4*a1 + 5*a2 + 1000, y > THRESHOLD)
reluplex.add(a1 == z1)                                   # stable on the box: phase fixed, no split

# one linear sub-problem per phase of relu(z2); push/pop undo the case after each check
for name, case in [("inactive", [z2 <= 0, a2 == 0]),
                   ("active  ", [z2 >= 0, a2 == z2])]:
    reluplex.push()
    reluplex.add(case)
    print(f"relu(z2) {name}: {reluplex.check()}")
    reluplex.pop()

The inactive branch is closed: there $y = 4z_1 + 1000 \le 4\cdot110 + 1000 = 1440$.
All the counterexamples live in the active branch.

### From *a* counterexample to the worst one

Z3 can also optimise (`z3.Optimize`, a Z3 extension). This asks a different question:
not *is there* a violation, but *how large* can $y$ get?

In [ ]:
opt = z3.Optimize()
opt.add(box + network)
opt.maximize(y)
opt.check()
mo = opt.model()
print(f"max y = {mo[y]}  at  rho = {mo[rho]}, v = {mo[v]}")

## Step 3b — the MILP (big-M)

Verification as optimisation: compute $y^* = \max\{\,y : \text{input in the box}\,\}$; the property holds
iff $y^* \le 1500$.

Each **unstable** ReLU $a = \mathrm{relu}(z)$, $z \in [l, u]$, becomes linear constraints plus one binary $p$:

$$a \ge 0,\qquad a \ge z,\qquad a \le z - l\,(1-p),\qquad a \le u\,p,\qquad p \in \{0,1\}.$$

A **stable** ReLU needs no binary: $a = z$ or $a = 0$. Here that means **one binary** in total:
$a_1 = z_1$ is substituted directly, and the variables are $x = (\rho, v, a_2, p)$.

`scipy.optimize.milp` wants the problem as $\min c^\top x$ s.t. $Ax \le b$, so every constraint
is moved to that form; the comment on each row is the constraint as you would write it on the board.

Replacing $p \in \{0,1\}$ by $p \in [0,1]$ gives the **LP relaxation**, the triangle of the slides.
The same function does both: only `integrality` changes.

In [ ]:
def max_y(rho_box, v_box, relax=False):
    (l1, u1), (l2, u2) = preactivation_bounds(rho_box, v_box)
    assert l1 >= 0 and l2 < 0 < u2, "this encoding assumes relu(z1) stable active, relu(z2) unstable"

    # variables  x = [rho, v, a2, p]
    # objective  y = 4·(rho + v − 10) + 5·a2 + 1000   ->  milp minimises, so we negate
    c = -np.array([4, 4, 5, 0])
    A = np.array([
        #  rho    v    a2    p
        [   2,  -30,  -1,    0],    # a2 >= z2              <=>   2·rho − 30·v − a2        <=  80
        [  -2,   30,   1,  -l2],    # a2 <= z2 − l2·(1 − p)  <=>  −2·rho + 30·v + a2 − l2·p <= −80 − l2
        [   0,    0,   1,  -u2],    # a2 <= u2·p            <=>   a2 − u2·p                 <=  0
    ])
    b = [80, -80 - l2, 0]
    bounds = Bounds([rho_box[0], v_box[0], 0,      0],          # a2 >= 0
                    [rho_box[1], v_box[1], np.inf, 1])          # p in [0, 1]
    integrality = [0, 0, 0, 0 if relax else 1]                  # p binary (MILP) or continuous (LP)

    res = milp(c, constraints=LinearConstraint(A, -np.inf, b), bounds=bounds, integrality=integrality)
    rho, v, a2, p = np.round(res.x, 9) + 0.0                    # + 0.0 turns -0.0 into 0.0
    return dict(y=4*(rho + v - 10) + 5*a2 + 1000, rho=rho, v=v, a2=a2, p=p)

mip = max_y(RHO, V)
print(f"MILP: y* = {mip['y']:g}  at  rho = {mip['rho']:g}, v = {mip['v']:g}   (p = {mip['p']:g})")
print("property holds" if mip["y"] <= THRESHOLD else f"y* > {THRESHOLD}: property VIOLATED")

### Bounds in Reluplex and in MILP

Both methods depend on the bounds $l, u$, in different ways. MILP needs them **before** solving:
they are the big-M constants, and they must be sound — a bound that is too tight cuts off real
points and the answer is wrong. Reluplex derives them **during** the search, and the property takes
part: $y > 1500$ is a bound and propagates backwards, which is how $a_2 \ge 12$ fixed `relu(z2)` above.

### The LP relaxation

In [ ]:
lp = max_y(RHO, V, relax=True)
print(f"LP:   y_LP = {lp['y']:g}  at  rho = {lp['rho']:g}, v = {lp['v']:g}   (p = {lp['p']:g})")

# is the LP point a real counterexample? run it through the network
_, y_check = net(lp["rho"], lp["v"])
print(f"re-run the LP point through the network: y = {y_check:g}")

The LP is **exact** here. Its optimum sits on the corner $\rho=100, v=0$, where $z_2 = u_2 = 120$, and
there the upper side of the triangle touches the ReLU. Its point, run through the network, is a
**real** counterexample, as the slide says: if $f(x_{LP})$ violates the property, you are done.

## Step 4 — does the property hold?

**No.** $y^* = 1960 > 1500$, reached at $\rho = 100$, $v = 0$.

| method | bound on $y$ | verdict |
|---|---|---|
| intervals | 2040 | inconclusive |
| LP relaxation | 1960 | violated: $x_{LP}$ is a real counterexample |
| MILP | 1960 (exact) | violated |
| Z3 | — | `sat`: a counterexample with $y = 1501$ |

## Comparing the three bounds

The three methods are ordered: $\;y^*_{\text{MILP}} \le y_{LP} \le y_{\text{interval}}$.
Who wins depends on the question. Same network, three questions:

In [ ]:
# interval, LP and MILP on one box and one threshold T, each with its verdict on "y <= T"
def compare(label, rho_box, v_box, T):
    bnds = preactivation_bounds(rho_box, v_box)
    y_int = C0 + sum(c * max(u, 0) for c, (l, u) in zip(C, bnds))
    lp, mip = max_y(rho_box, v_box, relax=True), max_y(rho_box, v_box)
    _, y_at_lp = net(lp["rho"], lp["v"])

    verdict_int  = "proved" if y_int <= T else "inconclusive"
    if lp["y"] <= T + 1e-6:        verdict_lp = "proved"
    elif y_at_lp > T:              verdict_lp = "violated (x_LP is a real counterexample)"
    else:                          verdict_lp = f"inconclusive (x_LP is not a real counterexample: f(x_LP) = {y_at_lp:.1f})"
    verdict_mip = "proved" if mip["y"] <= T + 1e-6 else "violated"

    print(f"{label}:  ρ ∈ {rho_box}, v ∈ {v_box}, property y ≤ {T}")
    print(f"  intervals  {y_int:8.1f}   {verdict_int}")
    print(f"  LP         {lp['y']:8.1f}   {verdict_lp}")
    print(f"  MILP       {mip['y']:8.1f}   {verdict_mip}\n")

compare("(a) the exercise", RHO, V, 1500)
compare("(b) LP proves what intervals cannot", RHO, V, 2000)
compare("(c) LP inconclusive, only MILP decides", (50, 90), (3, 16), 1390)

- **(a)** The exercise: LP and MILP agree, and the property is false.
- **(b)** A looser property ($y \le 2000$): intervals give 2040 and cannot decide. The LP
  gives 1960 and proves it, with no binaries.
- **(c)** A smaller box: the LP bound is just above the threshold, but its point is **not a real
  counterexample**: run through the network it does not violate the property. Only the MILP
  decides. This is the incompleteness of the relaxation.

### Questions

1. In (c), print `p` and `a2` at the LP optimum. Why is the point not a real counterexample?
2. Shrink the box until `relu(z2)` becomes stable too (the `assert` in `max_y` will fire).
   Write the encoding for that box: how many binaries? How does the LP bound compare with the MILP?
3. Replace `THRESHOLD = 1500` with the smallest threshold for which the property holds.
   Which of the methods above gives it to you directly?

## Going further: Marabou

Reluplex itself is no longer maintained. Its successor, from the same group, is **Marabou**: an
SMT-based solver with its own simplex core and a lazy search over the phases of the ReLUs, "in the
Reluplex spirit". The default configuration today also tightens the bounds with DeepPoly (Lecture 10)
and uses a different search procedure (DeepSoI), so it is not the 2017 algorithm line by line. It reads
networks in ONNX or NNet, and properties from Python or in VNN-LIB.

The query of this exercise, written directly in Reluplex form: bounds, equations, ReLU pairs.

```python
from maraboupy import MarabouNetwork, Marabou

def query(threshold):
    net = MarabouNetwork.MarabouNetwork()
    rho, v, z1, z2, a1, a2, y = (net.getNewVariable() for _ in range(7))

    # bounds: the input box and the NEGATED property  y >= threshold
    net.setLowerBound(rho, 50);  net.setUpperBound(rho, 100)
    net.setLowerBound(v, 0);     net.setUpperBound(v, 20)
    net.setLowerBound(y, threshold)

    # equations, as  sum(coeff * var) = scalar
    net.addEquality([z1, rho, v], [1, -1, -1], -10)    # z1 = rho + v - 10
    net.addEquality([z2, rho, v], [1, -2, 30], -80)    # z2 = 2 rho - 30 v - 80
    net.addEquality([y, a1, a2], [1, -4, -5], 1000)    # y  = 4 a1 + 5 a2 + 1000

    # ReLU pairs
    net.addRelu(z1, a1)
    net.addRelu(z2, a2)

    exit_code, vals, stats = net.solve(verbose=False, options=Marabou.createOptions(verbosity=0))
    print(f"y >= {threshold}: {exit_code}", end="")
    print(f"   rho = {vals[rho]:g}, v = {vals[v]:g}, y = {vals[y]:g}" if exit_code == "sat" else "")

query(1501)   # sat: the property y <= 1500 is false
query(2000)   # unsat: the property y < 2000 holds
```

```
y >= 1501: sat   rho = 67.2143, v = 0, y = 1501
y >= 2000: unsat
```

Bounds are non-strict, as in VNN-LIB: with `query(1500)` Marabou returns $\rho = 67.14$, $v = 0$ and
$y = 1500$ exactly, which does not violate $y \le 1500$. Hence the 1501.

**Installation is optional.** `pip install maraboupy` (version 2.0.0) has wheels only for Linux
(on Windows: WSL) and Intel Macs, with Python 3.8–3.11. It is not in `requirements.txt` and does not
install on Colab. Tested with maraboupy 2.0.0 and Python 3.11 on Linux.

*References:* G. Katz et al., *The Marabou framework for verification and analysis of deep neural
networks*, CAV 2019. H. Wu et al., *Marabou 2.0: a versatile formal analyzer of neural networks*,
CAV 2024. Code: github.com/NeuralNetworkVerification/Marabou